# Phase 0 — CSV → Parquet Conversion

**Notebook:** `notebooks/00_setup/03_csv_to_parquet.ipynb`

One-time conversion of raw CSV datasets into Parquet format. Two stages:

**Stage 1 (interim, `data/interim/`):** lossless. Strip column whitespace, replace inf with NaN, write Parquet.

**Stage 2 (processed, `data/processed/`):** analysis-ready. Drop NaN rows, encode labels (binary + categorical), enforce dtypes (float32 + int32), drop leakage columns (Flow ID, IPs, ports, Timestamp).

### Output schema contract (`data/processed/`)

Every processed Parquet has these columns:
- All feature columns as `float32`/`int32`, snake_case names
- `label` (`int8`): 0 = benign, 1 = attack
- `attack_category` (`category`): granular attack name (e.g. `'DoS Hulk'`, `'BENIGN'`)
- `attempted` (`bool`): True for Engelen `*-Attempted` rows (always False for non-CICIDS datasets)
- `source_file` (`category`): origin filename, for day-split experiments

### Scope

- CICIDS2017 (Engelen-improved): 5 day files
- UNSW-NB15: training + test partitions
- NSL-KDD: train + test partitions
- **Not converted:** CSE-CIC-IDS-2018 (34 GB, deferred until Phase 2 needs it)

Runs once. Takes 10–20 minutes. Don't close the tab.

## 1. Setup

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import sys
sys.path.insert(0, '/content/drive/MyDrive/phd_thesis')

%pip install -q pyarrow tqdm

Mounted at /content/drive


In [2]:
import re, time, json, datetime
from pathlib import Path
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

ROOT = Path('/content/drive/MyDrive/phd_thesis')
RAW = ROOT / 'data' / 'raw'
INTERIM = ROOT / 'data' / 'interim'
PROCESSED = ROOT / 'data' / 'processed'
REPORT_PATH = ROOT / 'reports' / 'advisor_notes' / 'conversion_report.md'

for p in [INTERIM, PROCESSED, REPORT_PATH.parent]:
    p.mkdir(parents=True, exist_ok=True)

# Conversion report buffer
REPORT = {
    'generated': datetime.datetime.now().isoformat(timespec='seconds'),
    'datasets': {},
}

def human_size(p: Path) -> str:
    n = p.stat().st_size
    for unit in ['B', 'KB', 'MB', 'GB']:
        if n < 1024:
            return f'{n:.1f}{unit}'
        n /= 1024
    return f'{n:.1f}TB'

print(f'RAW       = {RAW}')
print(f'INTERIM   = {INTERIM}')
print(f'PROCESSED = {PROCESSED}')

RAW       = /content/drive/MyDrive/phd_thesis/data/raw
INTERIM   = /content/drive/MyDrive/phd_thesis/data/interim
PROCESSED = /content/drive/MyDrive/phd_thesis/data/processed


## 2. Helpers

In [3]:
# Column-name normalization: 'Total Fwd Packets ' → 'total_fwd_packets'
def normalize_column(name: str) -> str:
    name = name.strip()
    # Replace special chars with underscore; collapse multiple underscores
    name = re.sub(r'[^A-Za-z0-9]+', '_', name)
    name = re.sub(r'_+', '_', name).strip('_')
    return name.lower()

# Columns that leak session identity — must be removed from processed data
LEAKAGE_COLUMN_PATTERNS = [
    'flow_id', 'src_ip', 'dst_ip', 'source_ip', 'destination_ip',
    'src_port', 'dst_port', 'source_port', 'destination_port',
    'timestamp', 'id',
    # UNSW-specific session identifiers
    'srcip', 'dstip', 'sport', 'dsport', 'stime', 'ltime',
]

def find_leakage_columns(columns: list[str]) -> list[str]:
    return [c for c in columns if c in LEAKAGE_COLUMN_PATTERNS]

def find_label_column(columns: list[str]) -> str:
    """Locate the label column by name. Raise if ambiguous."""
    candidates = [c for c in columns if c == 'label'] or \
                 [c for c in columns if 'label' in c]
    if not candidates:
        raise ValueError(f'No label column found in {columns[:20]}...')
    if len(candidates) > 1:
        # Prefer exact match over partial
        exact = [c for c in candidates if c == 'label']
        return exact[0] if exact else candidates[0]
    return candidates[0]

def downcast_numeric(df: pd.DataFrame) -> pd.DataFrame:
    """Downcast wide numeric columns to halve file size.

    float64 -> float32 always (acceptable precision loss for flow features).
    int64   -> int32 if values fit (they always do for IDS flow features).
    """
    for c in df.select_dtypes(include=['float64']).columns:
        df[c] = df[c].astype('float32')
    for c in df.select_dtypes(include=['int64']).columns:
        col_min, col_max = df[c].min(), df[c].max()
        if (col_min >= np.iinfo('int32').min) and (col_max <= np.iinfo('int32').max):
            df[c] = df[c].astype('int32')
    return df

## 3. Stage 1 — CICIDS2017 (Engelen-improved): CSV → interim Parquet

Lossless. Just normalize column names, replace inf with NaN, persist as Parquet.

In [4]:
src_dir = RAW / 'cicids2017_improved' / 'CICIDS2017_improved'
dst_dir = INTERIM / 'cicids2017'
dst_dir.mkdir(parents=True, exist_ok=True)

csv_files = sorted(src_dir.glob('*.csv'))
print(f'Converting {len(csv_files)} CICIDS2017 CSVs to interim Parquet...\n')

cic_interim_stats = []
for csv in csv_files:
    out = dst_dir / f'{csv.stem}.parquet'
    if out.exists():
        print(f'  [skip] {out.name} already exists ({human_size(out)})')
        df = pd.read_parquet(out, columns=None)
        cic_interim_stats.append({
            'file': csv.name, 'rows': len(df), 'cols': df.shape[1],
            'csv_mb': csv.stat().st_size / 1024**2,
            'parquet_mb': out.stat().st_size / 1024**2,
        })
        continue

    t0 = time.time()
    print(f'  reading {csv.name} ({human_size(csv)})...')
    df = pd.read_csv(csv, low_memory=False)
    df.columns = [normalize_column(c) for c in df.columns]
    df = df.replace([np.inf, -np.inf], np.nan)
    df['source_file'] = csv.stem
    df.to_parquet(out, compression='snappy', index=False)
    elapsed = time.time() - t0
    print(f'    -> {out.name}  ({human_size(out)})  shape={df.shape}  {elapsed:.1f}s')
    cic_interim_stats.append({
        'file': csv.name, 'rows': len(df), 'cols': df.shape[1],
        'csv_mb': csv.stat().st_size / 1024**2,
        'parquet_mb': out.stat().st_size / 1024**2,
    })

REPORT['datasets']['cicids2017_interim'] = cic_interim_stats
print(f'\nInterim total Parquet: {sum(s["parquet_mb"] for s in cic_interim_stats):.1f} MB')
print(f'Interim total CSV    : {sum(s["csv_mb"]     for s in cic_interim_stats):.1f} MB')

Converting 5 CICIDS2017 CSVs to interim Parquet...

  reading friday.csv (272.0MB)...
    -> friday.parquet  (90.9MB)  shape=(547557, 92)  22.4s
  reading monday.csv (198.2MB)...
    -> monday.parquet  (72.7MB)  shape=(371624, 92)  20.6s
  reading thursday.csv (180.7MB)...
    -> thursday.parquet  (63.4MB)  shape=(362076, 92)  14.1s
  reading tuesday.csv (170.1MB)...
    -> tuesday.parquet  (62.9MB)  shape=(322078, 92)  12.3s
  reading wednesday.csv (277.8MB)...
    -> wednesday.parquet  (89.4MB)  shape=(496641, 92)  18.3s

Interim total Parquet: 379.3 MB
Interim total CSV    : 1098.9 MB


## 4. Stage 2 — CICIDS2017: interim → processed

Apply schema contract: binary label, attempted flag, attack_category, drop leakage cols, drop NaN rows, downcast numeric dtypes.

In [5]:
src_dir = INTERIM / 'cicids2017'
dst_dir = PROCESSED / 'cicids2017'
dst_dir.mkdir(parents=True, exist_ok=True)

interim_files = sorted(src_dir.glob('*.parquet'))
cic_processed_stats = []

for interim in interim_files:
    out = dst_dir / interim.name
    if out.exists():
        print(f'  [skip] {out.name} already exists')
        df = pd.read_parquet(out)
        cic_processed_stats.append({
            'file': interim.name,
            'rows_in': None, 'rows_out': len(df),
            'rows_dropped_nan': None,
            'parquet_mb': out.stat().st_size / 1024**2,
            'class_balance': df['label'].value_counts().to_dict(),
            'attack_categories': df['attack_category'].value_counts().to_dict(),
        })
        continue

    t0 = time.time()
    print(f'  processing {interim.name}...')
    df = pd.read_parquet(interim)
    rows_in = len(df)

    # Locate columns
    label_col = find_label_column(df.columns.tolist())
    attempted_col = 'attempted_category' if 'attempted_category' in df.columns else None

    # Build attack_category (granular) BEFORE we munge label
    raw_labels = df[label_col].astype(str).str.strip()
    # Engelen attempted variants are encoded as 'DoS Hulk - Attempted' etc.
    attempted_mask = raw_labels.str.contains(' - Attempted', case=False, na=False)
    attack_category = raw_labels.str.replace(' - Attempted', '', regex=False)

    # Binary label: BENIGN = 0, everything else (including *-Attempted) = 1
    # (Decision: Attempted counts as Attack — recorded in conversion_report.md)
    binary_label = (attack_category.str.upper() != 'BENIGN').astype('int8')

    # Drop leakage columns + the original label column
    leakage = find_leakage_columns(df.columns.tolist())
    drop_cols = leakage + [label_col]
    if attempted_col:
        drop_cols.append(attempted_col)
    df = df.drop(columns=[c for c in drop_cols if c in df.columns])

    # Attach contract columns
    df['label'] = binary_label.values
    df['attack_category'] = pd.Categorical(attack_category.values)
    df['attempted'] = attempted_mask.values
    # source_file came from stage 1 — preserve it
    if 'source_file' in df.columns:
        df['source_file'] = df['source_file'].astype('category')

    # Drop rows with any NaN in feature columns (post-inf-to-NaN cleanup)
    contract_cols = ['label', 'attack_category', 'attempted', 'source_file']
    feature_cols = [c for c in df.columns if c not in contract_cols]
    before = len(df)
    df = df.dropna(subset=feature_cols)
    rows_dropped = before - len(df)

    # Downcast float64 -> float32 AND int64 -> int32 where safe
    df = downcast_numeric(df)

    # Persist
    df.to_parquet(out, compression='snappy', index=False)
    elapsed = time.time() - t0
    print(f'    -> {out.name}  ({human_size(out)})  '
          f'rows {rows_in:>7} -> {len(df):>7} (dropped {rows_dropped}) '
          f'cols={df.shape[1]}  {elapsed:.1f}s')

    cic_processed_stats.append({
        'file': interim.name,
        'rows_in': rows_in, 'rows_out': len(df),
        'rows_dropped_nan': rows_dropped,
        'parquet_mb': out.stat().st_size / 1024**2,
        'class_balance': df['label'].value_counts().to_dict(),
        'attack_categories': df['attack_category'].value_counts().head(20).to_dict(),
    })

REPORT['datasets']['cicids2017_processed'] = cic_processed_stats
print(f'\nProcessed total: {sum(s["parquet_mb"] for s in cic_processed_stats):.1f} MB')

  processing friday.parquet...
    -> friday.parquet  (62.3MB)  rows  547557 ->  547557 (dropped 0) cols=86  5.6s
  processing monday.parquet...
    -> monday.parquet  (49.1MB)  rows  371624 ->  371621 (dropped 3) cols=86  5.0s
  processing thursday.parquet...
    -> thursday.parquet  (42.6MB)  rows  362076 ->  362075 (dropped 1) cols=86  4.5s
  processing tuesday.parquet...
    -> tuesday.parquet  (42.4MB)  rows  322078 ->  322078 (dropped 0) cols=86  3.2s
  processing wednesday.parquet...
    -> wednesday.parquet  (62.0MB)  rows  496641 ->  496640 (dropped 1) cols=86  6.4s

Processed total: 258.5 MB


## 5. UNSW-NB15 — interim + processed

UNSW comes pre-split into training-set.csv and testing-set.csv. We process both partitions but preserve their split identity via `source_file`.

In [6]:
src_dir = RAW / 'unsw_nb15'
interim_dir = INTERIM / 'unsw_nb15'
processed_dir = PROCESSED / 'unsw_nb15'
interim_dir.mkdir(parents=True, exist_ok=True)
processed_dir.mkdir(parents=True, exist_ok=True)

# Find the training/testing partitions; UNSW Kaggle bundle naming varies slightly
all_csvs = list(src_dir.rglob('*.csv'))
train_files = [f for f in all_csvs if 'train' in f.name.lower()]
test_files = [f for f in all_csvs if 'test' in f.name.lower()]

print(f'Found UNSW CSVs: {[f.name for f in all_csvs]}')
print(f'  training: {[f.name for f in train_files]}')
print(f'  testing : {[f.name for f in test_files]}')

unsw_files = train_files + test_files
unsw_stats = []

for csv in unsw_files:
    # --- Stage 1: interim ---
    interim_out = interim_dir / f'{csv.stem}.parquet'
    if not interim_out.exists():
        print(f'\n  reading {csv.name} ({human_size(csv)})...')
        df = pd.read_csv(csv, low_memory=False)
        df.columns = [normalize_column(c) for c in df.columns]
        df = df.replace([np.inf, -np.inf], np.nan)
        df['source_file'] = csv.stem
        df.to_parquet(interim_out, compression='snappy', index=False)
        print(f'    interim -> {interim_out.name} ({human_size(interim_out)}) shape={df.shape}')
    else:
        print(f'  [skip interim] {interim_out.name}')
        df = pd.read_parquet(interim_out)

    # --- Stage 2: processed ---
    proc_out = processed_dir / f'{csv.stem}.parquet'
    if proc_out.exists():
        print(f'  [skip processed] {proc_out.name}')
        df_p = pd.read_parquet(proc_out)
        unsw_stats.append({
            'file': csv.name,
            'rows_out': len(df_p),
            'parquet_mb': proc_out.stat().st_size / 1024**2,
            'class_balance': df_p['label'].value_counts().to_dict(),
            'attack_categories': df_p['attack_category'].value_counts().head(20).to_dict(),
        })
        continue

    rows_in = len(df)
    # UNSW has both 'label' (binary 0/1) and 'attack_cat' (granular)
    label_col = find_label_column(df.columns.tolist())
    cat_col = None
    for c in ['attack_cat', 'attack_category']:
        if c in df.columns:
            cat_col = c
            break

    binary_label = df[label_col].astype('int8')
    if cat_col:
        attack_category = df[cat_col].astype(str).fillna('Normal').str.strip()
    else:
        attack_category = pd.Series(
            np.where(binary_label.values == 0, 'Normal', 'Unknown_Attack'),
            index=df.index,
        )

    # Drop leakage + original label/cat
    leakage = find_leakage_columns(df.columns.tolist())
    drop_cols = leakage + [label_col]
    if cat_col:
        drop_cols.append(cat_col)
    df = df.drop(columns=[c for c in drop_cols if c in df.columns])

    df['label'] = binary_label.values
    df['attack_category'] = pd.Categorical(attack_category.values)
    df['attempted'] = False  # UNSW has no attempted notion
    if 'source_file' in df.columns:
        df['source_file'] = df['source_file'].astype('category')

    contract_cols = ['label', 'attack_category', 'attempted', 'source_file']
    feature_cols = [c for c in df.columns if c not in contract_cols]
    # UNSW has categorical features (proto, service, state) — encode before NaN drop
    for c in feature_cols:
        if df[c].dtype == 'object':
            df[c] = pd.Categorical(df[c].fillna('unknown')).codes.astype('int32')
    before = len(df)
    df = df.dropna(subset=feature_cols)
    rows_dropped = before - len(df)

    df = downcast_numeric(df)
    df.to_parquet(proc_out, compression='snappy', index=False)
    print(f'    processed -> {proc_out.name} ({human_size(proc_out)}) '
          f'rows {rows_in} -> {len(df)} (dropped {rows_dropped})')

    unsw_stats.append({
        'file': csv.name,
        'rows_out': len(df),
        'parquet_mb': proc_out.stat().st_size / 1024**2,
        'class_balance': df['label'].value_counts().to_dict(),
        'attack_categories': df['attack_category'].value_counts().head(20).to_dict(),
    })

REPORT['datasets']['unsw_nb15'] = unsw_stats

Found UNSW CSVs: ['NUSW-NB15_features.csv', 'UNSW-NB15_1.csv', 'UNSW-NB15_2.csv', 'UNSW-NB15_3.csv', 'UNSW-NB15_LIST_EVENTS.csv', 'UNSW-NB15_4.csv', 'UNSW_NB15_training-set.csv', 'UNSW_NB15_testing-set.csv']
  training: ['UNSW_NB15_training-set.csv']
  testing : ['UNSW_NB15_testing-set.csv']

  reading UNSW_NB15_training-set.csv (14.7MB)...
    interim -> UNSW_NB15_training-set.parquet (6.5MB) shape=(82332, 46)
    processed -> UNSW_NB15_training-set.parquet (4.6MB) rows 82332 -> 82332 (dropped 0)

  reading UNSW_NB15_testing-set.csv (30.8MB)...
    interim -> UNSW_NB15_testing-set.parquet (13.4MB) shape=(175341, 46)
    processed -> UNSW_NB15_testing-set.parquet (9.9MB) rows 175341 -> 175341 (dropped 0)


## 6. NSL-KDD — interim + processed

NSL-KDD files have no header row; the column names are documented in the original KDD paper. We bake the schema in.

In [7]:
# Canonical NSL-KDD column names (41 features + label + difficulty)
NSL_COLS = [
    'duration', 'protocol_type', 'service', 'flag', 'src_bytes', 'dst_bytes',
    'land', 'wrong_fragment', 'urgent', 'hot', 'num_failed_logins', 'logged_in',
    'num_compromised', 'root_shell', 'su_attempted', 'num_root',
    'num_file_creations', 'num_shells', 'num_access_files', 'num_outbound_cmds',
    'is_host_login', 'is_guest_login', 'count', 'srv_count', 'serror_rate',
    'srv_serror_rate', 'rerror_rate', 'srv_rerror_rate', 'same_srv_rate',
    'diff_srv_rate', 'srv_diff_host_rate', 'dst_host_count',
    'dst_host_srv_count', 'dst_host_same_srv_rate', 'dst_host_diff_srv_rate',
    'dst_host_same_src_port_rate', 'dst_host_srv_diff_host_rate',
    'dst_host_serror_rate', 'dst_host_srv_serror_rate',
    'dst_host_rerror_rate', 'dst_host_srv_rerror_rate',
    'attack_label', 'difficulty',
]

src_dir = RAW / 'nsl_kdd'
interim_dir = INTERIM / 'nsl_kdd'
processed_dir = PROCESSED / 'nsl_kdd'
interim_dir.mkdir(parents=True, exist_ok=True)
processed_dir.mkdir(parents=True, exist_ok=True)

# NSL-KDD comes with multiple file formats; we want KDDTrain+ and KDDTest+
nsl_files = []
for pattern in ['KDDTrain+.txt', 'KDDTrain+.arff', 'KDDTest+.txt', 'KDDTest+.arff']:
    for f in src_dir.rglob(pattern):
        if f.suffix == '.txt':
            nsl_files.append(f)

# Deduplicate by stem (some Kaggle mirrors have multiple copies)
seen = set()
deduped = []
for f in nsl_files:
    if f.stem not in seen:
        deduped.append(f)
        seen.add(f.stem)
nsl_files = deduped

print(f'NSL-KDD files to convert: {[f.name for f in nsl_files]}')

nsl_stats = []
for txt in nsl_files:
    interim_out = interim_dir / f'{txt.stem}.parquet'
    if not interim_out.exists():
        print(f'\n  reading {txt.name} ({human_size(txt)})...')
        df = pd.read_csv(txt, header=None, names=NSL_COLS, low_memory=False)
        df = df.replace([np.inf, -np.inf], np.nan)
        df['source_file'] = txt.stem
        df.to_parquet(interim_out, compression='snappy', index=False)
        print(f'    interim -> {interim_out.name} ({human_size(interim_out)}) shape={df.shape}')
    else:
        print(f'  [skip interim] {interim_out.name}')
        df = pd.read_parquet(interim_out)

    proc_out = processed_dir / f'{txt.stem}.parquet'
    if proc_out.exists():
        print(f'  [skip processed] {proc_out.name}')
        df_p = pd.read_parquet(proc_out)
        nsl_stats.append({
            'file': txt.name,
            'rows_out': len(df_p),
            'parquet_mb': proc_out.stat().st_size / 1024**2,
            'class_balance': df_p['label'].value_counts().to_dict(),
            'attack_categories': df_p['attack_category'].value_counts().head(20).to_dict(),
        })
        continue

    rows_in = len(df)
    raw_labels = df['attack_label'].astype(str).str.strip()
    binary_label = (raw_labels.str.lower() != 'normal').astype('int8')
    attack_category = raw_labels.copy()
    attack_category[binary_label == 0] = 'Normal'

    df = df.drop(columns=['attack_label', 'difficulty'])
    # Encode the 3 string-typed features
    for c in ['protocol_type', 'service', 'flag']:
        df[c] = pd.Categorical(df[c].astype(str).fillna('unknown')).codes.astype('int32')

    df['label'] = binary_label.values
    df['attack_category'] = pd.Categorical(attack_category.values)
    df['attempted'] = False
    if 'source_file' in df.columns:
        df['source_file'] = df['source_file'].astype('category')

    contract_cols = ['label', 'attack_category', 'attempted', 'source_file']
    feature_cols = [c for c in df.columns if c not in contract_cols]
    before = len(df)
    df = df.dropna(subset=feature_cols)
    rows_dropped = before - len(df)

    df = downcast_numeric(df)
    df.to_parquet(proc_out, compression='snappy', index=False)
    print(f'    processed -> {proc_out.name} ({human_size(proc_out)}) '
          f'rows {rows_in} -> {len(df)} (dropped {rows_dropped})')

    nsl_stats.append({
        'file': txt.name,
        'rows_out': len(df),
        'parquet_mb': proc_out.stat().st_size / 1024**2,
        'class_balance': df['label'].value_counts().to_dict(),
        'attack_categories': df['attack_category'].value_counts().head(20).to_dict(),
    })

REPORT['datasets']['nsl_kdd'] = nsl_stats

NSL-KDD files to convert: ['KDDTrain+.txt', 'KDDTest+.txt']

  reading KDDTrain+.txt (18.2MB)...
    interim -> KDDTrain+.parquet (2.3MB) shape=(125973, 44)
    processed -> KDDTrain+.parquet (2.2MB) rows 125973 -> 125973 (dropped 0)

  reading KDDTest+.txt (3.3MB)...
    interim -> KDDTest+.parquet (477.7KB) shape=(22544, 44)
    processed -> KDDTest+.parquet (464.8KB) rows 22544 -> 22544 (dropped 0)


## 7. Write the conversion report

Audit trail of what was filtered out and why. Goes into `reports/advisor_notes/` so it's preserved alongside the data card. Cite this in your thesis methods chapter.

In [8]:
lines = ['# CSV → Parquet Conversion Report\n',
        f'Generated: {REPORT["generated"]}\n']

lines.append('## Conversion decisions\n')
lines.append('- **Binary label encoding:** BENIGN/Normal → 0, all attacks (including '
             'Engelen `*-Attempted` variants) → 1. The granular category is preserved '
             'in `attack_category`; the attempted flag is preserved in `attempted`.\n')
lines.append('- **Leakage columns dropped from processed:** Flow ID, Src/Dst IP, '
             'Src/Dst Port, Timestamp, ID, and UNSW equivalents (srcip, dstip, sport, '
             'dsport, stime, ltime). These are session identifiers that would let a '
             'model memorize specific flows.\n')
lines.append('- **NaN handling:** rows with any NaN in feature columns are dropped. '
             '`inf` values are converted to NaN first (then dropped).\n')
lines.append('- **Dtype:** float64 -> float32, int64 -> int32 where safe. '
             'Categorical strings encoded as int32 codes.\n')
lines.append('- **Schema contract (every processed file):** features as float32/int32, '
             '`label` int8 {0,1}, `attack_category` categorical, `attempted` bool, '
             '`source_file` categorical.\n')

for dataset_name, stats in REPORT['datasets'].items():
    lines.append(f'\n## {dataset_name}\n')
    if not stats:
        lines.append('  (no files processed)\n')
        continue
    for s in stats:
        lines.append(f'\n### {s["file"]}\n')
        for k, v in s.items():
            if k == 'file':
                continue
            if isinstance(v, dict):
                lines.append(f'- {k}:\n')
                for sub_k, sub_v in v.items():
                    lines.append(f'    - `{sub_k}`: {sub_v}\n')
            elif isinstance(v, float):
                lines.append(f'- {k}: {v:.2f}\n')
            else:
                lines.append(f'- {k}: {v}\n')

REPORT_PATH.write_text(''.join(lines))
print(f'Written: {REPORT_PATH} ({REPORT_PATH.stat().st_size} bytes)')

# Also dump the structured JSON for programmatic access
json_path = REPORT_PATH.with_suffix('.json')
with open(json_path, 'w') as f:
    json.dump(REPORT, f, indent=2, default=str)
print(f'Written: {json_path}')

Written: /content/drive/MyDrive/phd_thesis/reports/advisor_notes/conversion_report.md (4666 bytes)
Written: /content/drive/MyDrive/phd_thesis/reports/advisor_notes/conversion_report.json


## 8. Final summary

In [9]:
print('=' * 64)
print('CONVERSION SUMMARY')
print('=' * 64)

for root, label in [(INTERIM, 'INTERIM'), (PROCESSED, 'PROCESSED')]:
    print(f'\n{label}:  {root}')
    for sub in sorted(root.iterdir()):
        if sub.is_dir():
            files = list(sub.rglob('*.parquet'))
            total = sum(f.stat().st_size for f in files)
            print(f'  {sub.name:25s} {len(files):3d} parquet files  '
                  f'{total/1024**2:>8.1f} MB')

print('\nNext: run notebooks/00_setup/04_verify_loaders.ipynb to verify the loaders.')

CONVERSION SUMMARY

INTERIM:  /content/drive/MyDrive/phd_thesis/data/interim
  cicids2017                  5 parquet files     379.3 MB
  nsl_kdd                     2 parquet files       2.7 MB
  unsw_nb15                   2 parquet files      19.8 MB

PROCESSED:  /content/drive/MyDrive/phd_thesis/data/processed
  cicids2017                  5 parquet files     258.5 MB
  nsl_kdd                     2 parquet files       2.7 MB
  unsw_nb15                   2 parquet files      14.5 MB

Next: run notebooks/00_setup/04_verify_loaders.ipynb to verify the loaders.
